### PyTorch 4: Bilder - Übungen

Im letzten Notebook der Reihe erkennt ein Netz **handgeschriebene Ziffern** aus dem berühmten Datensatz **MNIST**: 70 000 Bilder mit 28 × 28 Pixeln.
Zuerst mit einem Netz aus voll verbundenen Schichten wie in Notebook 3, dann mit einem **Faltungsnetz** (*convolutional neural network*, CNN),
der Art von Netz, mit der Bilderkennung in der Praxis arbeitet. Zum Schluss trainierst du auf der Grafikkarte und überträgst alles auf Bilder von Kleidungsstücken.

Die Reihe hat vier Notebooks:

1. Tensoren: erzeugen, umformen, auswählen, rechnen
2. Lernen: Gradienten, Gradientenabstieg, die Trainingsschleife
3. Netze: neuronale Netze bauen und trainieren, Mini-Batches, Überanpassung
4. **Bilder**: Ziffern erkennen mit voll verbundenen Netzen und Faltungsnetzen (dieses Notebook)

Vorausgesetzt wird *PyTorch: Netze* (Trainingsschleife, `DataLoader`).

**Dieses Notebook läuft nicht im Browser-Python der Übungsseite**, weil es PyTorch dort nicht gibt. Öffne es über den Link in der
Übersicht in **Google Colab** (dafür brauchst du ein Google-Konto). PyTorch ist in Colab schon installiert. Colab öffnet eine Kopie:
Wer seine Lösungen behalten will, wählt *Datei → Kopie in Drive speichern*.

Bei Zufallszahlen und beim Training können deine Ergebnisse in den letzten Stellen von den Lösungen abweichen, je nach PyTorch-Version und Rechner.
Aufgaben mit Training haben deshalb keine erwartete Ausgabe, sondern eine Beispiellösung und eine Angabe, ungefähr welche Werte herauskommen sollten.

Jeder Abschnitt beginnt mit kurzen Erklärungen und Beispielen. Führe die Beispiele aus und probiere sie ruhig ein wenig aus.
Danach folgen viele kleine Übungen. Wenn etwas vorgegeben ist, steht es schon in der Codezelle.

Unter den Aufgaben kannst du jeweils die Lösung aufklappen. Versuche es aber immer zuerst selbst!
Bei den Aufgaben „Was wird ausgegeben?“ überlegst du zuerst, führst dann die Zelle aus und vergleichst.
Die aufklappbare Lösung erklärt dort, wie die Ausgabe zustande kommt.

#### MNIST herunterladen

**Führe diese Zelle zuerst aus.** Sie lädt die MNIST-Daten (etwa 12 MB) in den Ordner `daten` herunter. In Colab bleiben sie dort, bis die Sitzung endet;
danach lädt die Zelle sie neu.

In [ ]:
from torchvision import datasets

datasets.MNIST(root='daten', train=True, download=True)
datasets.MNIST(root='daten', train=False, download=True)
print('MNIST ist bereit.')

## 1. Die MNIST-Daten

#### Bilder als Tensoren

`datasets.MNIST(root='daten', train=True, transform=transforms.ToTensor())` aus dem Paket `torchvision` sind die 60 000 **Trainingsbilder**, mit `train=False`
die 10 000 **Testbilder**. `transforms.ToTensor()` macht aus jedem Bild einen Tensor mit Helligkeiten von 0 (schwarz) bis 1 (weiß).

- `len(training)` ist die Anzahl der Bilder, `training[i]` das Paar `(bild, ziffer)`.
- Ein Bild hat die Form `[1, 28, 28]`: 1 **Kanal** (Graustufen; Farbbilder hätten 3 Kanäle für Rot, Grün, Blau) mit 28 × 28 Pixeln.
- `training.targets` enthält alle Ziffern als Tensor.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())

print(len(training), len(test))
bild, ziffer = training[0]
print(bild.shape, bild.dtype, ziffer)
print(f'{bild.min().item():.1f} {bild.max().item():.1f}')
print(training.targets[:10])

#### Bilder anzeigen

`plt.imshow(bild[0].numpy(), cmap='gray')` zeigt den einen Kanal eines Bilds an (`bild[0]` hat die Form `[28, 28]`). Wie in Notebook 1 wird der Tensor
für matplotlib vorher mit `.numpy()` in ein numpy-Array umgewandelt.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 2))
for i in range(10):
    bild, ziffer = training[i]
    plt.subplot(1, 10, i + 1)
    plt.imshow(bild[0].numpy(), cmap='gray')
    plt.title(ziffer)
    plt.axis('off')
plt.show()

#### Aufgabe 1.1

Gib für das Testbild mit Index 0 die Ziffer, die Form des Bilds und die mittlere Helligkeit aus (4 Nachkommastellen).

**Erwartete Ausgabe:**
```
7
torch.Size([1, 28, 28])
0.0923
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
bild, ziffer = test[0]
print(ziffer)
print(bild.shape)
print(f'{bild.mean().item():.4f}')
```

</details>

#### Aufgabe 1.2

Wie oft kommt jede Ziffer in den Trainingsdaten vor? Gib `torch.bincount(training.targets)` aus.

**Erwartete Ausgabe:**
```
tensor([5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949])
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
print(torch.bincount(training.targets))
```

</details>

#### Aufgabe 1.3

Gib die Ziffern der Trainingsbilder 100 bis 119 aus und die Indizes der ersten fünf Trainingsbilder, die eine 7 zeigen (`torch.where(training.targets == 7)[0][:5]`).

**Erwartete Ausgabe:**
```
tensor([5, 7, 1, 7, 1, 1, 6, 3, 0, 2, 9, 3, 1, 1, 0, 4, 9, 2, 0, 0])
tensor([15, 29, 38, 42, 52])
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
print(training.targets[100:120])
print(torch.where(training.targets == 7)[0][:5])
```

</details>

#### Aufgabe 1.4

Wie viele Pixel des ersten Trainingsbilds sind „Tinte“ (Helligkeit größer als 0.5)? Und welcher Anteil der Fläche ist das (3 Nachkommastellen)?

**Erwartete Ausgabe:**
```
111
0.142
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
bild, ziffer = training[0]
tinte = (bild > 0.5).sum().item()
print(tinte)
print(f'{tinte / bild.numel():.3f}')
```

</details>

#### Aufgabe 1.5

Zeige die ersten 8 Testbilder, die eine 3 zeigen, nebeneinander an.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import matplotlib.pyplot as plt
indizes = torch.where(test.targets == 3)[0][:8]
for nr, i in enumerate(indizes):
    plt.subplot(1, 8, nr + 1)
    plt.imshow(test[i.item()][0][0].numpy(), cmap='gray')
    plt.axis('off')
plt.show()
```

</details>

#### Aufgabe 1.6

Berechne das „mittlere Bild“ jeder Ziffer aus den ersten 10 000 Trainingsbildern und zeige die zehn Bilder nebeneinander an. Die Bilder als Tensor `[60000, 28, 28]`
mit Werten von 0 bis 255 erhältst du mit `training.data`; teile durch 255.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import matplotlib.pyplot as plt
bilder = training.data[:10000].float() / 255
ziffern = training.targets[:10000]
plt.figure(figsize=(10, 2))
for z in range(10):
    plt.subplot(1, 10, z + 1)
    plt.imshow(bilder[ziffern == z].mean(dim=0).numpy(), cmap='gray')
    plt.title(z)
    plt.axis('off')
plt.show()
```

</details>

#### Aufgabe 1.7

Was wird ausgegeben? Überlege zuerst, was ausgegeben wird. Führe dann die Zelle aus und vergleiche.

In [ ]:
import torch

bilder = torch.zeros(64, 1, 28, 28)
print(bilder.shape[0])
print(bilder.flatten(start_dim=1).shape)
print(bilder[0][0].shape)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
64
torch.Size([64, 784])
torch.Size([28, 28])
```

- Ein Stapel aus 64 Bildern: Die erste Dimension ist die Anzahl, `64`.
- `flatten(start_dim=1)` lässt die erste Dimension stehen und macht aus jedem Bild 784 Zahlen: `torch.Size([64, 784])`.
- `bilder[0]` ist ein Bild `[1, 28, 28]`, `bilder[0][0]` sein einziger Kanal: `torch.Size([28, 28])`.

</details>

## 2. Ein voll verbundenes Netz

#### 784 Eingaben

Ein Netz aus `Linear`-Schichten braucht Vektoren: `nn.Flatten()` macht aus jedem Bild `[1, 28, 28]` einen Vektor mit 784 Zahlen. Danach folgen eine verborgene Schicht
und 10 Ausgaben, eine pro Ziffer. Jedes Neuron der ersten Schicht ist mit **allen** 784 Pixeln verbunden (**voll verbunden**).

Die Daten kommen über einen `DataLoader` in Portionen von 64 Bildern. Damit es auch ohne Grafikkarte schnell geht, wird hier nur mit den ersten 10 000 Trainingsbildern
trainiert (`Subset(training, range(10000))`). Die Funktionen `trainiere()` und `genauigkeit()` stehen in den Aufgaben schon in der Zelle; sie schieben die Daten auch
auf die Grafikkarte, falls es eine gibt (Abschnitt 5).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
print(modell)
print(sum([p.numel() for p in modell.parameters()]))
xb, yb = next(iter(lader))
print(xb.shape, modell(xb).shape)
trainiere(modell, lader, 2)
print(f'Test-Genauigkeit: {genauigkeit(modell, testlader):.3f}')

#### Aufgabe 2.1

Rechne die Anzahl der Gewichte des Netzes `Flatten`, `Linear(784, 128)`, `ReLU`, `Linear(128, 10)` von Hand aus und vergleiche mit PyTorch.

**Erwartete Ausgabe:**
```
101770
101770
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
print(784 * 128 + 128 + 128 * 10 + 10)
modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
print(sum([p.numel() for p in modell.parameters()]))
```

</details>

#### Aufgabe 2.2

Hole mit `next(iter(testlader))` die erste Portion aus den Testdaten (in der Zelle: Portionen zu 1000 Bildern) und gib die Formen der Bilder und der Ziffern
und die Form der Ausgabe eines (untrainierten) Netzes aus.

**Erwartete Ausgabe:**
```
torch.Size([1000, 1, 28, 28]) torch.Size([1000])
torch.Size([1000, 10])
```

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

xb, yb = next(iter(testlader))
print(xb.shape, yb.shape)
modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
print(modell(xb).shape)
```

</details>

#### Aufgabe 2.3

Ein untrainiertes Netz rät. Gib seine Genauigkeit auf den Testdaten aus (Seed 1). Sie sollte etwa 0.1 sein (eine von zehn Ziffern).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

torch.manual_seed(1)
modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
print(f'{genauigkeit(modell, testlader):.3f}')
```

</details>

#### Aufgabe 2.4

Trainiere das Netz aus dem Beispiel 1, 3 und 5 Epochen lang (jeweils neu, Seed 0) und gib die Epochen und die Test-Genauigkeit aus. Sie sollte von etwa 0.90 auf etwa 0.93 steigen.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

for epochen in [1, 3, 5]:
    torch.manual_seed(0)
    modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
    trainiere(modell, lader, epochen)
    print(epochen, f'{genauigkeit(modell, testlader):.3f}')
```

</details>

#### Aufgabe 2.5

Trainiere das Netz aus dem Beispiel (Seed 0, 2 Epochen) und zeige die ersten 10 Testbilder mit der Vorhersage als Titel an. Die Vorhersage für ein einzelnes Bild bekommst du
mit `modell(bild.unsqueeze(0).to(device)).argmax().item()` (das Bild braucht eine Stapel-Dimension).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt

torch.manual_seed(0)
modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
trainiere(modell, lader, 2)
modell.eval()
plt.figure(figsize=(10, 2))
for i in range(10):
    bild, ziffer = test[i]
    with torch.no_grad():
        vorhersage = modell(bild.unsqueeze(0).to(device)).argmax().item()
    plt.subplot(1, 10, i + 1)
    plt.imshow(bild[0].numpy(), cmap='gray')
    plt.title(vorhersage)
    plt.axis('off')
plt.show()
```

</details>

#### Aufgabe 2.6

Was wird ausgegeben? Überlege zuerst, was ausgegeben wird. Führe dann die Zelle aus und vergleiche.

In [ ]:
import torch
from torch import nn

modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
print(modell(torch.zeros(5, 1, 28, 28)).shape)
print(len(modell))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
torch.Size([5, 10])
4
```

- 5 Bilder rein, für jedes 10 Ausgaben: `torch.Size([5, 10])`.
- Das Netz besteht aus 4 Schichten: `Flatten`, `Linear`, `ReLU`, `Linear`: `4`.

</details>

## 3. Faltungsnetze

#### Filter, die über das Bild gleiten

Ein voll verbundenes Netz weiß nicht, welche Pixel nebeneinanderliegen. Ein **Faltungsnetz** nutzt das aus: Ein kleiner **Filter** (z. B. 3 × 3 Gewichte) gleitet über das Bild.
An jeder Stelle wird die gewichtete Summe der 3 × 3 Pixel darunter berechnet. So entsteht ein neues Bild (eine **Merkmalskarte**), das zeigt, wo das Muster des Filters
vorkommt, z. B. senkrechte Kanten. Die Gewichte der Filter lernt das Netz selbst.

Das Beispiel zeigt einen von Hand gewählten Filter für senkrechte Kanten: links negative, rechts positive Gewichte. `F.conv2d(bilder, filter, padding=1)` aus
`torch.nn.functional` wendet ihn an (`padding=1` ergänzt einen Rand aus Nullen, damit das Bild 28 × 28 bleibt).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import torch.nn.functional as F
import matplotlib.pyplot as plt

bild, ziffer = training[0]
filter = torch.tensor([[[[-1.0, 0.0, 1.0],
                         [-1.0, 0.0, 1.0],
                         [-1.0, 0.0, 1.0]]]])     # Form [1, 1, 3, 3]: 1 Filter, 1 Kanal
karte = F.conv2d(bild.unsqueeze(0), filter, padding=1)
print(karte.shape)
plt.subplot(1, 2, 1)
plt.imshow(bild[0].numpy(), cmap='gray')
plt.subplot(1, 2, 2)
plt.imshow(karte[0, 0].numpy(), cmap='bwr')
plt.show()

#### `Conv2d` und `MaxPool2d`

- `nn.Conv2d(1, 16, 3, padding=1)`: aus 1 Eingabekanal werden 16 Merkmalskarten, jede mit einem eigenen gelernten 3 × 3-Filter.
- `nn.MaxPool2d(2)` halbiert Breite und Höhe: Von je 2 × 2 Pixeln bleibt der größte Wert. Das macht das Netz unempfindlicher gegen kleine Verschiebungen.

Das Netz unten verkleinert das Bild so von 28 × 28 auf 14 × 14 und dann 7 × 7 mit 32 Merkmalskarten. Diese 32 · 7 · 7 = 1568 Zahlen gehen in eine `Linear`-Schicht mit
10 Ausgaben. Faltungsnetze brauchen viel weniger Gewichte als voll verbundene und sind trotzdem besser.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
x = torch.zeros(1, 1, 28, 28)
for schicht in modell:
    x = schicht(x)
    print(type(schicht).__name__, list(x.shape))
print(sum([p.numel() for p in modell.parameters()]))
torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
trainiere(modell, lader, 2)
print(f'Test-Genauigkeit: {genauigkeit(modell, testlader):.3f}')

#### Aufgabe 3.1

Gib die Form der Ausgabe von `nn.Conv2d(1, 8, 3)` (ohne `padding`) und von `nn.Conv2d(1, 8, 3, padding=1)` für ein Bild `torch.zeros(1, 1, 28, 28)` aus. Ohne Rand
wird das Bild an jeder Seite um ein Pixel kleiner.

**Erwartete Ausgabe:**
```
torch.Size([1, 8, 26, 26])
torch.Size([1, 8, 28, 28])
```

In [ ]:
import torch
from torch import nn
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
x = torch.zeros(1, 1, 28, 28)
print(nn.Conv2d(1, 8, 3)(x).shape)
print(nn.Conv2d(1, 8, 3, padding=1)(x).shape)
```

</details>

#### Aufgabe 3.2

Wie viele Gewichte hat `nn.Conv2d(1, 16, 3)`? Jeder der 16 Filter hat 3 · 3 Gewichte und einen bias-Wert. Rechne es aus und vergleiche mit PyTorch. Wie viele
hat `nn.Conv2d(16, 32, 3)` (jeder Filter hat jetzt 16 · 3 · 3 Gewichte)?

**Erwartete Ausgabe:**
```
160 160
4640 4640
```

In [ ]:
import torch
from torch import nn
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
print(16 * 3 * 3 + 16, sum([p.numel() for p in nn.Conv2d(1, 16, 3).parameters()]))
print(32 * 16 * 3 * 3 + 32, sum([p.numel() for p in nn.Conv2d(16, 32, 3).parameters()]))
```

</details>

#### Aufgabe 3.3

Wende `nn.MaxPool2d(2)` auf die 4 × 4-Matrix `m` an (als Tensor der Form `[1, 1, 4, 4]`) und gib das Ergebnis aus. Von jedem 2 × 2-Block bleibt das Maximum.

**Erwartete Ausgabe:**
```
tensor([[[[4., 2.],
          [2., 7.]]]])
```

In [ ]:
import torch
from torch import nn

m = torch.tensor([[1.0, 3.0, 2.0, 0.0], [4.0, 2.0, 1.0, 1.0], [0.0, 1.0, 5.0, 6.0], [2.0, 2.0, 7.0, 3.0]])
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn

m = torch.tensor([[1.0, 3.0, 2.0, 0.0], [4.0, 2.0, 1.0, 1.0], [0.0, 1.0, 5.0, 6.0], [2.0, 2.0, 7.0, 3.0]])
print(nn.MaxPool2d(2)(m.reshape(1, 1, 4, 4)))
```

</details>

#### Aufgabe 3.4

Wende den Kantenfilter aus dem Beispiel von Hand auf das kleine Bild `b` an (`F.conv2d` ohne `padding`). Das Bild ist links dunkel und rechts hell; der Filter findet
die Kante in der Mitte.

**Erwartete Ausgabe:**
```
tensor([[[[3., 3.]]]])
```

In [ ]:
import torch
import torch.nn.functional as F

b = torch.tensor([[0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0]]).reshape(1, 1, 3, 4)
filter = torch.tensor([[[[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]]]])
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
import torch.nn.functional as F

b = torch.tensor([[0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0]]).reshape(1, 1, 3, 4)
filter = torch.tensor([[[[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]]]])
print(F.conv2d(b, filter))
```

</details>

#### Aufgabe 3.5

Ändere den Filter aus dem Beispiel so, dass er **waagerechte** Kanten findet (oben −1, unten 1), und zeige das erste Trainingsbild und die Merkmalskarte nebeneinander.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import torch.nn.functional as F
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
import torch.nn.functional as F
import matplotlib.pyplot as plt
bild, ziffer = training[0]
filter = torch.tensor([[[[-1.0, -1.0, -1.0],
                         [0.0, 0.0, 0.0],
                         [1.0, 1.0, 1.0]]]])
karte = F.conv2d(bild.unsqueeze(0), filter, padding=1)
plt.subplot(1, 2, 1)
plt.imshow(bild[0].numpy(), cmap='gray')
plt.subplot(1, 2, 2)
plt.imshow(karte[0, 0].numpy(), cmap='bwr')
plt.show()
```

</details>

#### Aufgabe 3.6

Vergleiche die Anzahl der Gewichte des voll verbundenen Netzes und des Faltungsnetzes aus den Beispielen und trainiere beide (Seed 0, 2 Epochen). Gib für jedes den Namen,
die Anzahl der Gewichte und die Test-Genauigkeit aus. Das Faltungsnetz hat nur etwa ein Fünftel der Gewichte und ist besser (etwa 0.95 gegen 0.91).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

for name in ['voll verbunden', 'Faltung']:
    torch.manual_seed(0)
    if name == 'Faltung':
        modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
    else:
        modell = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10))
    trainiere(modell, lader, 2)
    print(name, sum([p.numel() for p in modell.parameters()]), f'{genauigkeit(modell, testlader):.3f}')
```

</details>

#### Aufgabe 3.7

Zeige die 16 gelernten Filter der ersten Schicht des trainierten Faltungsnetzes (Seed 0, 2 Epochen) als kleine Bilder an: `modell[0].weight` hat die Form `[16, 1, 3, 3]`,
`modell[0].weight[i, 0].detach().cpu()` ist der Filter `i`.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt

torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
trainiere(modell, lader, 2)
filter = modell[0].weight.detach().cpu()
plt.figure(figsize=(8, 2))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(filter[i, 0].numpy(), cmap='bwr')
    plt.axis('off')
plt.show()
```

</details>

#### Aufgabe 3.8

Was wird ausgegeben? Überlege zuerst, was ausgegeben wird. Führe dann die Zelle aus und vergleiche.

In [ ]:
import torch
from torch import nn

x = torch.zeros(10, 3, 32, 32)
print(nn.Conv2d(3, 6, 5)(x).shape)
print(nn.MaxPool2d(2)(x).shape)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
torch.Size([10, 6, 28, 28])
torch.Size([10, 3, 16, 16])
```

- 10 Farbbilder (3 Kanäle) mit 32 × 32 Pixeln. Ein 5 × 5-Filter ohne Rand macht das Bild an jeder Seite 2 Pixel kleiner: 32 − 4 = 28.
  Aus 3 Kanälen werden 6 Merkmalskarten: `torch.Size([10, 6, 28, 28])`.
- MaxPool halbiert Breite und Höhe, die Kanäle bleiben: `torch.Size([10, 3, 16, 16])`.

</details>

## 4. Fehler ansehen, speichern und laden

#### Ein Modell speichern

Das Training dauert; danach will man das Netz behalten. `modell.state_dict()` enthält alle Gewichte, `torch.save(modell.state_dict(), 'ziffern.pt')` speichert sie in einer Datei.
Zum Laden erzeugt man ein Netz mit **demselben Aufbau** und lädt die Gewichte mit `modell.load_state_dict(torch.load('ziffern.pt'))`.

In Colab liegt die Datei im Ordner der Sitzung (links das Ordner-Symbol); dort kann man sie auch herunterladen.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
trainiere(modell, lader, 1)
torch.save(modell.state_dict(), 'ziffern.pt')

neues_modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
neues_modell.load_state_dict(torch.load('ziffern.pt'))
print(f'{genauigkeit(modell, testlader):.3f} {genauigkeit(neues_modell.to(device), testlader):.3f}')

#### Aufgabe 4.1

Gib die Namen und Formen aller Einträge im `state_dict()` des Faltungsnetzes aus (`for name, w in modell.state_dict().items()`). Die Namen bestehen aus der Nummer der
Schicht und `weight` bzw. `bias`.

**Erwartete Ausgabe:**
```
0.weight [16, 1, 3, 3]
0.bias [16]
3.weight [32, 16, 3, 3]
3.bias [32]
7.weight [10, 1568]
7.bias [10]
```

In [ ]:
import torch
from torch import nn
# your code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
for name, w in modell.state_dict().items():
    print(name, list(w.shape))
```

</details>

#### Aufgabe 4.2

Trainiere das Faltungsnetz (Seed 0, 2 Epochen) und zeige die ersten 10 Testbilder an, die es falsch erkennt, mit dem Titel `richtig→vorhergesagt`.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import matplotlib.pyplot as plt

torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
trainiere(modell, lader, 2)
modell.eval()
xb, yb = next(iter(testlader))
with torch.no_grad():
    vorhersage = modell(xb.to(device)).argmax(dim=1).cpu()
falsch = torch.where(vorhersage != yb)[0][:10]
plt.figure(figsize=(10, 2))
for nr, i in enumerate(falsch):
    plt.subplot(1, 10, nr + 1)
    plt.imshow(xb[i, 0].numpy(), cmap='gray')
    plt.title(f'{yb[i].item()}→{vorhersage[i].item()}')
    plt.axis('off')
plt.show()
```

</details>

#### Aufgabe 4.3

Welche Ziffern erkennt das Faltungsnetz (Seed 0, 2 Epochen) am schlechtesten? Berechne für jede Ziffer den Anteil der richtig erkannten Testbilder dieser Ziffer
(3 Nachkommastellen). Sammle dafür alle Vorhersagen der Testdaten mit `torch.cat()`.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
trainiere(modell, lader, 2)
modell.eval()
vorhersagen = []
with torch.no_grad():
    for xb, yb in testlader:
        vorhersagen.append(modell(xb.to(device)).argmax(dim=1).cpu())
vorhersagen = torch.cat(vorhersagen)
for z in range(10):
    auswahl = test.targets == z
    print(z, f'{(vorhersagen[auswahl] == z).float().mean().item():.3f}')
```

</details>

#### Aufgabe 4.4

Zeichne selbst eine Ziffer: Erzeuge ein Bild `torch.zeros(28, 28)` und setze Bereiche mit Slicing auf 1, z. B. für eine 1 die Spalten 13 bis 15 in den Zeilen 5 bis 23.
Lass das gespeicherte Netz aus dem Beispiel (`ziffern.pt`, vorher das Beispiel ausführen) die Ziffer erkennen und gib die drei wahrscheinlichsten Ziffern mit
`torch.softmax` und `topk(3)` aus.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
import matplotlib.pyplot as plt
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
modell.load_state_dict(torch.load('ziffern.pt'))
modell.eval()
bild = torch.zeros(28, 28)
bild[5:23, 13:16] = 1.0
with torch.no_grad():
    p = torch.softmax(modell(bild.reshape(1, 1, 28, 28)), dim=1)[0]
werte, ziffern = p.topk(3)
for w, z in zip(werte, ziffern):
    print(z.item(), f'{w.item():.3f}')
plt.imshow(bild.numpy(), cmap='gray')
plt.show()
```

</details>

## 5. Mehr Rechenleistung: die Grafikkarte

#### Alle 60 000 Bilder

Mit der Grafikkarte lohnt es sich, mit **allen** Trainingsbildern und mehr Epochen zu trainieren. Stelle in Colab den Laufzeittyp auf *T4 GPU* um
(*Laufzeit → Laufzeittyp ändern*) und führe danach die Setup-Zelle noch einmal aus. Die Funktionen `trainiere()` und `genauigkeit()` schieben Netz und Daten mit
`.to(device)` auf die Grafikkarte. Ohne Grafikkarte läuft der Code auch, dauert aber deutlich länger.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import time

print(device)
alle = DataLoader(training, batch_size=128, shuffle=True)
torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
start = time.perf_counter()
trainiere(modell, alle, 1)
print(f'{time.perf_counter() - start:.1f} s')
print(f'Test-Genauigkeit: {genauigkeit(modell, testlader):.3f}')

#### Aufgabe 5.1

Trainiere das Faltungsnetz mit allen Trainingsbildern 5 Epochen lang auf der Grafikkarte und gib nach jeder Epoche die Test-Genauigkeit aus. Sie sollte auf etwa 0.99 steigen.
(Ohne Grafikkarte dauert das einige Minuten.)

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

alle = DataLoader(training, batch_size=128, shuffle=True)
torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
for epoche in range(1, 6):
    trainiere(modell, alle, 1)
    print(epoche, f'{genauigkeit(modell, testlader):.3f}')
```

</details>

#### Aufgabe 5.2

Miss, wie lange eine Epoche mit allen Trainingsbildern auf der CPU und auf der Grafikkarte dauert. Setze dazu `device` einmal auf `'cpu'` und einmal auf `'cuda'`
(die Funktionen benutzen die Variable `device`).

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import time
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
import time

alle = DataLoader(training, batch_size=128, shuffle=True)
geraete = ['cpu', 'cuda'] if torch.cuda.is_available() else ['cpu']
for geraet in geraete:
    device = geraet
    torch.manual_seed(0)
    modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
    start = time.perf_counter()
    trainiere(modell, alle, 1)
    print(geraet, f'{time.perf_counter() - start:.1f} s')
```

</details>

## 6. Gemischte Übungen

#### Aufgabe 6.1

**Fashion-MNIST** hat dieselbe Form wie MNIST, zeigt aber Kleidungsstücke (0 T-Shirt, 1 Hose, 2 Pullover, 3 Kleid, 4 Mantel, 5 Sandale, 6 Hemd, 7 Sneaker,
8 Tasche, 9 Stiefel). Lade es mit `datasets.FashionMNIST(...)` (gleiche Parameter wie MNIST), zeige die ersten 10 Bilder mit ihrer Klasse an und trainiere das
Faltungsnetz darauf (10 000 Bilder, Seed 0, 3 Epochen). Die Test-Genauigkeit sollte bei etwa 0.84 liegen: Kleidung ist schwerer als Ziffern.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
namen = ['T-Shirt', 'Hose', 'Pullover', 'Kleid', 'Mantel', 'Sandale', 'Hemd', 'Sneaker', 'Tasche', 'Stiefel']
training = datasets.FashionMNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.FashionMNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
plt.figure(figsize=(12, 2))
for i in range(10):
    bild, klasse = training[i]
    plt.subplot(1, 10, i + 1)
    plt.imshow(bild[0].numpy(), cmap='gray')
    plt.title(namen[klasse], fontsize=8)
    plt.axis('off')
plt.show()
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)
modell = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
verlustfunktion = nn.CrossEntropyLoss()
optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
for epoche in range(3):
    for xb, yb in lader:
        verlust = verlustfunktion(modell(xb), yb)
        optimierer.zero_grad()
        verlust.backward()
        optimierer.step()
richtig = 0
with torch.no_grad():
    for xb, yb in DataLoader(test, batch_size=1000):
        richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
print(f'{richtig / len(test):.3f}')
```

</details>

#### Aufgabe 6.2

Gegen Überanpassung bei Bildern hilft **Dropout**: `nn.Dropout(0.25)` setzt beim Training zufällig ein Viertel der Werte auf 0 (bei `modell.eval()` nicht).
Baue ein Faltungsnetz mit `nn.Dropout(0.25)` vor der letzten `Linear`-Schicht, trainiere es wie im Beispiel (Seed 0, 2 Epochen) und vergleiche die Test-Genauigkeit mit dem
Netz ohne Dropout. Bei so kurzem Training macht Dropout noch kaum einen Unterschied (etwa 0.95 in beiden Fällen); es hilft vor allem bei langem Training
und großen Netzen.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

for name in ['ohne', 'mit Dropout']:
    torch.manual_seed(0)
    schichten = [nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                 nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten()]
    if name == 'mit Dropout':
        schichten.append(nn.Dropout(0.25))
    schichten.append(nn.Linear(32 * 7 * 7, 10))
    modell = nn.Sequential(*schichten)
    trainiere(modell, lader, 2)
    print(name, f'{genauigkeit(modell, testlader):.3f}')
```

</details>

#### Aufgabe 6.3

Baue ein eigenes, größeres Faltungsnetz (z. B. drei `Conv2d`-Schichten oder mehr Filter) und versuche, mit den 10 000 Trainingsbildern in 3 Epochen eine möglichst hohe
Test-Genauigkeit zu erreichen. Mehr als 0.97 ist gut.

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)
# your code

<details>
<summary>Beispiellösung anzeigen (erst selbst probieren!)</summary>

```python
import torch
from torch import nn
from torchvision import datasets, transforms

training = datasets.MNIST(root='daten', train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root='daten', train=False, download=True, transform=transforms.ToTensor())
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
lader = DataLoader(Subset(training, range(10000)), batch_size=64, shuffle=True)   # 10 000 Bilder, damit es schnell geht
testlader = DataLoader(test, batch_size=1000)

def trainiere(modell, lader, epochen):
    modell.to(device)
    verlustfunktion = nn.CrossEntropyLoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=0.001)
    for epoche in range(epochen):
        modell.train()
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            verlust = verlustfunktion(modell(xb), yb)
            optimierer.zero_grad()
            verlust.backward()
            optimierer.step()

def genauigkeit(modell, lader):
    modell.eval()
    richtig = 0
    with torch.no_grad():
        for xb, yb in lader:
            xb, yb = xb.to(device), yb.to(device)
            richtig += (modell(xb).argmax(dim=1) == yb).sum().item()
    return richtig / len(lader.dataset)

torch.manual_seed(0)
modell = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Dropout(0.25), nn.Linear(64 * 7 * 7, 10))
trainiere(modell, lader, 3)
print(f'{genauigkeit(modell, testlader):.3f}')
```

</details>